In [1]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from newspaper import Article
from tqdm import tqdm

data = pd.read_csv("data_wrangling_part2.csv")

In [3]:
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m
27352,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,0.000174,0.037050,0.171840
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161


This week is data modeling but for my project theres a few things I have to do: Come up with the best way to scrape the test from from the articles, analyze how many articles were lost in the process and redo the encoding for publisher size and checking the minimum article cutoff. Additionally I need to apply FinBert to get the sentiment analysis for the text of each article. Using the text instead of the headline makes sense because it is more representative of what the author is actually saying even at the cost of losing half of the data set due to paywalls and old articles not existing any more. Lastly, I need to determine the best model, how im splitting to handle time as well as being able to classify articles, and how im going to use the model to assign all the publishers a score. 

I used AI to help me make my article downloads (get_text_from_article function from last week) faster with using concurrent features and threading so it does not take 3 days to add the article text colummn. I am going to scrape all the articles, analyze how many rows actually have text, and go from there.  Still have to figure out why I get blocked sometimes from scraping.

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_text_from_article(url):
    try:
        article = Article(url)
        article.download()  
        article.parse()
        text = article.text
        return text if len(text) >= 1 else ""
    except Exception:
        return ""

results = [""] * len(data)

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = {
        executor.submit(get_text_from_article, url): i
        for i, url in enumerate(data["url"])
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        results[idx] = future.result()

data["article_text"] = results

100%|██████████| 27357/27357 [1:06:30<00:00,  6.86it/s]


In [6]:
data.tail()

,headline,url,publisher,published_at,ticker,Headline_Class_Prediction,total_article_count,publisher_size,return_7d,return_30d,return_6m,article_text
27352,UPDATE: J.P. Morgan Raises Price Target on Yum...,https://www.benzinga.com/analyst-ratings/analy...,Steven Anfield,2011-12-08,YUM,0,30,0,0.000174,0.037050,0.171840,
27353,JP Morgan Raises PT on Yum Brands to $68,https://www.benzinga.com/analyst-ratings/price...,Joe Young,2011-12-08,YUM,0,401,0,0.000174,0.037050,0.171840,
27354,YUM! Cites Dividends and Buybacks as Ways to R...,https://www.benzinga.com/news/11/12/2183552/yu...,Matthew Kennedy,2011-12-07,YUM,0,76,0,-0.011033,0.031718,0.160814,
27355,"Jefferies Reiterates Hold, $55 Target on Yum! ...",https://www.benzinga.com/analyst-ratings/analy...,Phil Marsh,2011-12-06,YUM,0,99,0,0.016571,0.036455,0.133400,
27356,Yum! Brands Announces Full-Year 2012 Expectati...,https://www.benzinga.com/news/11/12/2174970/yu...,Matthew Kennedy,2011-12-05,YUM,1,76,0,0.032756,0.032930,0.135161,


In [15]:
data.to_csv('temp_scrape.csv', index=False) # Dont want to wait a 3rd time for this scrape once it actually works

In [14]:
scraped_articles = data['article_text'].fillna("").astype(str).str.strip().ne("").sum()
print(scraped_articles)

218


Scraping did not work at all so trying a different library

In [4]:
import requests
import trafilatura
from concurrent.futures import ThreadPoolExecutor, as_completed
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
}

def get_article_text(url):
    if pd.isna(url) or not str(url).strip():
        return ""

    try:
        response = requests.get(str(url).strip(), headers=headers, timeout=15)
        response.raise_for_status()

        text = trafilatura.extract(
            response.text,
            favor_precision=True,
            include_comments=False,
            include_tables=False
        )

        return text.strip() if text else ""

    except Exception:
        return ""


In [ ]:
results = [""] * len(data)

with ThreadPoolExecutor(max_workers=6) as executor:
    futures = {
        executor.submit(get_article_text, url): i
        for i, url in enumerate(data["url"])
    }

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx = futures[future]
        results[idx] = future.result()

data["article_text"] = results

  7%|▋         | 1862/27357 [04:13<57:54,  7.34it/s]  


In [26]:
data.to_csv('temp_scrape.csv', index=False) # My wifi cut out at 83% 

This scraping is going to take a while so im leaving my notes here: I need to check scrape performance, if its fine then remove all the rows that are empty, reanalyze the author threshold + encoding, FinbertAnalysis, then test a simple model.

In [ ]:
# Check performance
scraped_articles = data['article_text'].fillna("").astype(str).str.strip().ne("").sum()
print(scraped_articles)

KeyError: 'article_text'

In [ ]:
# Check some of the text in the rows

In [ ]:
# Remove Rows without text

In [ ]:
len(data)

In [ ]:
data["publisher"].value_counts().describe() # Check publisher counts

In [ ]:
publisher_amount = data['publisher'].value_counts()
publisher_cutoff = publisher_amount[publisher_amount >= 25].index
data = data[data['publisher'].isin(publisher_cutoff)].copy()
len(data)

In [ ]:
data['total_article_count'].describe()  # Rencode based on new values

In [ ]:
# REMEMBER TO UPDATE THESE VALUES
def encode_article_counts(publisher):
    if publisher <= 413:
        return 0
    elif publisher <= 2428:
        return 1
    else:
        return 2
data['publisher_size'] = data['total_article_count'].apply(encode_article_counts)
data.tail()

In [ ]:
# Finbert